# Modalities Checkpoint Perplexity on FineWeb-Edu (sample-10BT)

Loads a Modalities checkpoint and computes perplexity on a deterministic, fixed token budget drawn from `HuggingFaceFW/fineweb-edu` (`sample-10BT`).

The token buffer is built deterministically (fixed stream order + seed) and cached to disk so every checkpoint scores the **same exact tokens** — perplexities are directly comparable across runs.

Reports:
* main-head perplexity (`output['logits']`)
* per-recurrence perplexities (`output['each_recurrence_logits']`, if available)

## Run with ai2-olmes env !!!!!!!

In [1]:
import sys; print(sys.executable)

/raid/s3/opengptx/behzad_shomali/modalities/olmes/.venv/bin/python


In [2]:
import os
import sys
import math
import tempfile
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm

WORKSPACE_ROOT = Path('/raid/s3/opengptx/behzad_shomali')
MODALITIES_SRC = WORKSPACE_ROOT / 'modalities' / 'src'
if str(MODALITIES_SRC) not in sys.path:
    sys.path.insert(0, str(MODALITIES_SRC))

from modalities.evaluation.olmes_evaluator import load_modalities_model

/raid/s3/opengptx/behzad_shomali/modalities/olmes/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/raid/s3/opengptx/behzad_shomali/modalities/olmes/.venv/lib/python3.11/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
Skipping import of cpp extensions due to incompatible torch version 2.8.0+cu128 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
2026-05-12:18:17:17,210 INFO     [rouge_scorer.py:83] Using default tokenizer.


## Config

In [3]:
# ----- Checkpoint -----
CHECKPOINT_PATH = "/raid/s3/opengptx/behzad_shomali/checkpoints/2026-05-06__02-32-43_d7e0557d0359234b/eid_2026-05-06__02-32-43_d7e0557d0359234b-seen_steps_27864-seen_tokens_6847856640-target_steps_27868-target_tokens_6848839680"
CONFIG_PATH = None    # non-DCP: path to config yaml. DCP: None.
MODEL_KEY = 'model_raw'

# ----- Device / context -----
DEVICE = 'cuda:7' if torch.cuda.is_available() else 'cpu'
MAX_CONTEXT_TOKENS = 2048
BATCH_SIZE = 4  # forward pass micro-batch (windows per fwd)

# ----- Evaluation budget (kept fixed across runs for comparability) -----
N_TOKENS_BUDGET = 5_000_000
STREAM_SEED = 42
DATASET_NAME = 'HuggingFaceFW/fineweb-edu'
DATASET_CONFIG = 'sample-10BT'
DATASET_SPLIT = 'train'
TEXT_KEY = 'text'

# ----- Token-buffer cache (so every run uses the same tokens) -----
TOKEN_CACHE_DIR = Path('/raid/s3/opengptx/behzad_shomali/modalities/loop_MTP_paper/.ppl_cache')
TOKEN_CACHE_DIR.mkdir(parents=True, exist_ok=True)
# The cache key encodes everything that affects the token sequence.
# It does NOT include the checkpoint — the same buffer is reused across checkpoints.
TOKEN_CACHE_FILE = TOKEN_CACHE_DIR / (
    f'fwedu_{DATASET_CONFIG}_n{N_TOKENS_BUDGET}_seed{STREAM_SEED}.npy'
)

# ----- MTP / per-recurrence -----
REPORT_PER_RECURRENCE = True

print('DEVICE =', DEVICE)
print('CHECKPOINT_PATH set:', bool(CHECKPOINT_PATH) and os.path.exists(CHECKPOINT_PATH))
print('Token cache file:', TOKEN_CACHE_FILE)

DEVICE = cuda:7
CHECKPOINT_PATH set: True
Token cache file: /raid/s3/opengptx/behzad_shomali/modalities/loop_MTP_paper/.ppl_cache/fwedu_sample-10BT_n5000000_seed42.npy


## Load model & tokenizer

In [4]:
assert CHECKPOINT_PATH and os.path.exists(CHECKPOINT_PATH), \
    f'Set CHECKPOINT_PATH to an existing checkpoint. Got: {CHECKPOINT_PATH!r}'

temp_conversion_dir = tempfile.mkdtemp(prefix='modalities_converted_')

model, tokenizer, loaded_config = load_modalities_model(
    checkpoint_path=CHECKPOINT_PATH,
    config_path=CONFIG_PATH,
    model_key=MODEL_KEY,
    converted_output_dir=temp_conversion_dir,
)
model = model.to(DEVICE)
model.eval()
print('Model loaded.')

Detected distributed checkpoint (DCP) at: /raid/s3/opengptx/behzad_shomali/checkpoints/2026-05-06__02-32-43_d7e0557d0359234b/eid_2026-05-06__02-32-43_d7e0557d0359234b-seen_steps_27864-seen_tokens_6847856640-target_steps_27868-target_tokens_6848839680
Converting DCP to PyTorch format...
Converted config: /tmp/modalities_converted_331ftxuz/3MTP.yaml
Instantiated <class 'modalities.checkpointing.torch.torch_checkpoint_loading.TorchCheckpointLoading'>: checkpointed_model -> config -> checkpoint_loading
Building block 0 of type BlockTypes.GROUP_RECURSIVE_MTP
Using WS for combining representations in CombinedRepresentationGPT2Block
Instantiated <class 'modalities.models.gpt2.gpt2_model.GPT2LLM'>: model
Model loaded with 260349706 trainable parameters from /tmp/modalities_converted_331ftxuz/pytorch_model.bin
Instantiated <class 'modalities.models.gpt2.gpt2_model.GPT2LLM'>: checkpointed_model
Loading tokenizer from original DCP config...
Loading tokenizer...
No tokenizer found in config. Falli

## Build (or load cached) token buffer

Streams FineWeb-Edu sample-10BT with a fixed seed and tokenizes documents until `N_TOKENS_BUDGET` tokens are collected. Adds an EOS between docs if the tokenizer exposes one. The buffer is cached as a uint32 numpy array on disk.

In [5]:
def _get_eos_id(tok):
    hf = tok.tokenizer if hasattr(tok, 'tokenizer') else tok
    for attr in ('eos_token_id', 'eot_token_id'):
        v = getattr(hf, attr, None)
        if v is not None:
            return int(v)
    return None

def build_token_buffer(tokenizer, n_tokens: int, seed: int) -> np.ndarray:
    from datasets import load_dataset

    ds = load_dataset(
        DATASET_NAME, name=DATASET_CONFIG, split=DATASET_SPLIT, streaming=True
    )
    # Deterministic order: shuffle the stream with a fixed seed and small buffer.
    ds = ds.shuffle(seed=seed, buffer_size=10_000)

    eos_id = _get_eos_id(tokenizer)
    buf = np.empty(n_tokens, dtype=np.uint32)
    filled = 0
    pbar = tqdm(total=n_tokens, desc='tokenizing', unit='tok')
    for row in ds:
        text = row.get(TEXT_KEY, '')
        if not text:
            continue
        ids = tokenizer.tokenize(text)
        if eos_id is not None:
            ids = list(ids) + [eos_id]
        remaining = n_tokens - filled
        take = min(len(ids), remaining)
        buf[filled:filled + take] = ids[:take]
        filled += take
        pbar.update(take)
        if filled >= n_tokens:
            break
    pbar.close()
    assert filled == n_tokens, f'only collected {filled}/{n_tokens} tokens'
    return buf

if TOKEN_CACHE_FILE.exists():
    token_buffer = np.load(TOKEN_CACHE_FILE)
    print(f'Loaded cached token buffer: {TOKEN_CACHE_FILE} ({len(token_buffer):,} tokens)')
else:
    token_buffer = build_token_buffer(tokenizer, N_TOKENS_BUDGET, STREAM_SEED)
    np.save(TOKEN_CACHE_FILE, token_buffer)
    print(f'Built and cached token buffer at {TOKEN_CACHE_FILE} ({len(token_buffer):,} tokens)')

assert len(token_buffer) == N_TOKENS_BUDGET

Loaded cached token buffer: /raid/s3/opengptx/behzad_shomali/modalities/loop_MTP_paper/.ppl_cache/fwedu_sample-10BT_n5000000_seed42.npy (5,000,000 tokens)


## Split into non-overlapping windows

In [6]:
# Each window is MAX_CONTEXT_TOKENS long. The final partial window is dropped
# so the scored token count is exactly the same across runs.
n_windows = len(token_buffer) // MAX_CONTEXT_TOKENS
windows = token_buffer[: n_windows * MAX_CONTEXT_TOKENS].reshape(n_windows, MAX_CONTEXT_TOKENS)
windows_t = torch.from_numpy(windows.astype(np.int64))

# Per window we score MAX_CONTEXT_TOKENS - 1 positions (next-token prediction).
scored_tokens_per_window = MAX_CONTEXT_TOKENS - 1
total_scored_tokens = n_windows * scored_tokens_per_window
print(f'n_windows = {n_windows}')
print(f'scored tokens = {total_scored_tokens:,}')

n_windows = 2441
scored tokens = 4,996,727


## Compute perplexity

In [7]:
model.need_mtp_logits = True


def forward_outputs(input_ids: torch.Tensor) -> dict:
    with torch.no_grad():
        out = model({'input_ids': input_ids})
    return out["logits"]

def _ce_sum(logits: torch.Tensor, targets: torch.Tensor) -> float:
    # logits: (B, T, V). Predict targets[:, 1:] from logits[:, :-1].
    shift_logits = logits[:, :-1, :].contiguous().float()
    shift_targets = targets[:, 1:].contiguous()
    loss = F.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_targets.view(-1),
        reduction='sum',
    )
    return loss.item()

# Detect whether per-recurrence logits are available & meaningful for this checkpoint.
with torch.no_grad():
    probe = forward_outputs(windows_t[:1].to(DEVICE))
has_per_recurrence = (
    REPORT_PER_RECURRENCE
    and 'each_recurrence_logits' in probe
    and probe['each_recurrence_logits'] is not None
    and probe['each_recurrence_logits'].shape[0] > 1
    # Filler values are exactly -1000 (see gpt2_model.py); detect by checking max.
    and probe['each_recurrence_logits'][-1].max().item() > -999.0
)
n_recurrences = int(probe['each_recurrence_logits'].shape[0]) if has_per_recurrence else 0
print(f'per-recurrence reporting: {has_per_recurrence}  (n={n_recurrences})')
del probe
torch.cuda.empty_cache()

per-recurrence reporting: True  (n=4)


In [8]:
nll_main_sum = 0.0
nll_rec_sum = [0.0] * n_recurrences if has_per_recurrence else []

for start in tqdm(range(0, n_windows, BATCH_SIZE), desc='ppl'):
    batch = windows_t[start:start + BATCH_SIZE].to(DEVICE, non_blocking=True)
    out = forward_outputs(batch)

    nll_main_sum += _ce_sum(out['logits'], batch)

    if has_per_recurrence:
        rec_logits = out['each_recurrence_logits']  # (K, B, T, V) — [0]=combined/main, [1..]=per iter
        for r in range(n_recurrences):
            nll_rec_sum[r] += _ce_sum(rec_logits[r], batch)

    del out

ppl_main = math.exp(nll_main_sum / total_scored_tokens)
print(f'\n=== Perplexity on {DATASET_NAME}/{DATASET_CONFIG}, {total_scored_tokens:,} tokens ===')
print(f'main:        NLL/tok = {nll_main_sum / total_scored_tokens:.4f}   PPL = {ppl_main:.3f}')

if has_per_recurrence:
    for r in range(n_recurrences):
        nll_tok = nll_rec_sum[r] / total_scored_tokens
        label = 'combined[0]' if r == 0 else f'recurrence[{r}]'
        print(f'{label:>14s}: NLL/tok = {nll_tok:.4f}   PPL = {math.exp(nll_tok):.3f}')

ppl: 100%|██████████| 611/611 [03:42<00:00,  2.74it/s]


=== Perplexity on HuggingFaceFW/fineweb-edu/sample-10BT, 4,996,727 tokens ===
main:        NLL/tok = 2.9848   PPL = 19.783
   combined[0]: NLL/tok = 2.9848   PPL = 19.783
 recurrence[1]: NLL/tok = 16.6232   PPL = 16571084.984
 recurrence[2]: NLL/tok = 19.1310   PPL = 203469456.633
 recurrence[3]: NLL/tok = 12.7074   PPL = 330184.853


## Summary dict (handy to log)

In [9]:
summary = {
    'checkpoint': CHECKPOINT_PATH,
    'dataset': f'{DATASET_NAME}/{DATASET_CONFIG}',
    'n_scored_tokens': int(total_scored_tokens),
    'context_len': int(MAX_CONTEXT_TOKENS),
    'seed': int(STREAM_SEED),
    'ppl_main': float(ppl_main),
    'nll_main_per_tok': float(nll_main_sum / total_scored_tokens),
}
if has_per_recurrence:
    summary['ppl_per_recurrence'] = [
        float(math.exp(s / total_scored_tokens)) for s in nll_rec_sum
    ]
summary

{'checkpoint': '/raid/s3/opengptx/behzad_shomali/checkpoints/2026-05-06__02-32-43_d7e0557d0359234b/eid_2026-05-06__02-32-43_d7e0557d0359234b-seen_steps_27864-seen_tokens_6847856640-target_steps_27868-target_tokens_6848839680',
 'dataset': 'HuggingFaceFW/fineweb-edu/sample-10BT',
 'n_scored_tokens': 4996727,
 'context_len': 2048,
 'seed': 42,
 'ppl_main': 19.7831062653119,
 'nll_main_per_tok': 2.984828354578407,
 'ppl_per_recurrence': [19.7831062653119,
  16571084.983678173,
  203469456.63322565,
  330184.8527376172]}

## OpenWebText (Skylion007/openwebtext)

Same pipeline as above, but on `Skylion007/openwebtext`. The token buffer is cached separately so both datasets always score the same exact tokens across runs.

In [10]:
# Generalized buffer builder (parameterized by dataset).
def build_token_buffer_from(
    tokenizer,
    n_tokens: int,
    seed: int,
    dataset_name: str,
    dataset_config = None,
    dataset_split: str = "train",
    text_key: str = "text",
) -> np.ndarray:
    from datasets import load_dataset

    ds = load_dataset(
        dataset_name, name=dataset_config, split=dataset_split, streaming=True
    )
    ds = ds.shuffle(seed=seed, buffer_size=10_000)

    eos_id = _get_eos_id(tokenizer)
    buf = np.empty(n_tokens, dtype=np.uint32)
    filled = 0
    pbar = tqdm(total=n_tokens, desc=f'tokenizing[{dataset_name}]', unit='tok')
    for row in ds:
        text = row.get(text_key, '')
        if not text:
            continue
        ids = tokenizer.tokenize(text)
        if eos_id is not None:
            ids = list(ids) + [eos_id]
        remaining = n_tokens - filled
        take = min(len(ids), remaining)
        buf[filled:filled + take] = ids[:take]
        filled += take
        pbar.update(take)
        if filled >= n_tokens:
            break
    pbar.close()
    assert filled == n_tokens, f'only collected {filled}/{n_tokens} tokens'
    return buf


OWT_DATASET_NAME = 'Skylion007/openwebtext'
OWT_DATASET_CONFIG = None
OWT_TOKEN_CACHE_FILE = TOKEN_CACHE_DIR / (
    f'openwebtext_n{N_TOKENS_BUDGET}_seed{STREAM_SEED}.npy'
)

if OWT_TOKEN_CACHE_FILE.exists():
    owt_token_buffer = np.load(OWT_TOKEN_CACHE_FILE)
    print(f'Loaded cached token buffer: {OWT_TOKEN_CACHE_FILE} ({len(owt_token_buffer):,} tokens)')
else:
    owt_token_buffer = build_token_buffer_from(
        tokenizer,
        n_tokens=N_TOKENS_BUDGET,
        seed=STREAM_SEED,
        dataset_name=OWT_DATASET_NAME,
        dataset_config=OWT_DATASET_CONFIG,
    )
    np.save(OWT_TOKEN_CACHE_FILE, owt_token_buffer)
    print(f'Built and cached token buffer at {OWT_TOKEN_CACHE_FILE} ({len(owt_token_buffer):,} tokens)')

assert len(owt_token_buffer) == N_TOKENS_BUDGET

tokenizing[Skylion007/openwebtext]: 100%|██████████| 5000000/5000000 [00:09<00:00, 526395.61tok/s]

Built and cached token buffer at /raid/s3/opengptx/behzad_shomali/modalities/loop_MTP_paper/.ppl_cache/openwebtext_n5000000_seed42.npy (5,000,000 tokens)


In [11]:
# Split OWT buffer into non-overlapping windows
owt_n_windows = len(owt_token_buffer) // MAX_CONTEXT_TOKENS
owt_windows = owt_token_buffer[: owt_n_windows * MAX_CONTEXT_TOKENS].reshape(owt_n_windows, MAX_CONTEXT_TOKENS)
owt_windows_t = torch.from_numpy(owt_windows.astype(np.int64))

owt_total_scored_tokens = owt_n_windows * (MAX_CONTEXT_TOKENS - 1)
print(f'OWT n_windows = {owt_n_windows}')
print(f'OWT scored tokens = {owt_total_scored_tokens:,}')

OWT n_windows = 2441
OWT scored tokens = 4,996,727


In [12]:
# Compute perplexity on OpenWebText (main head + per-recurrence if available)
owt_nll_main_sum = 0.0
owt_nll_rec_sum = [0.0] * n_recurrences if has_per_recurrence else []

for start in tqdm(range(0, owt_n_windows, BATCH_SIZE), desc='ppl[owt]'):
    batch = owt_windows_t[start:start + BATCH_SIZE].to(DEVICE, non_blocking=True)
    out = forward_outputs(batch)
    if isinstance(out, dict):
        main_logits = out['logits']
    else:
        main_logits = out
    owt_nll_main_sum += _ce_sum(main_logits, batch)

    if has_per_recurrence and isinstance(out, dict):
        rec_logits = out['each_recurrence_logits']
        for r in range(n_recurrences):
            owt_nll_rec_sum[r] += _ce_sum(rec_logits[r], batch)

    del out

owt_ppl_main = math.exp(owt_nll_main_sum / owt_total_scored_tokens)
print(f'\n=== Perplexity on {OWT_DATASET_NAME}, {owt_total_scored_tokens:,} tokens ===')
print(f'main:        NLL/tok = {owt_nll_main_sum / owt_total_scored_tokens:.4f}   PPL = {owt_ppl_main:.3f}')

if has_per_recurrence:
    for r in range(n_recurrences):
        nll_tok = owt_nll_rec_sum[r] / owt_total_scored_tokens
        label = 'combined[0]' if r == 0 else f'recurrence[{r}]'
        print(f'{label:>14s}: NLL/tok = {nll_tok:.4f}   PPL = {math.exp(nll_tok):.3f}')

ppl[owt]: 100%|██████████| 611/611 [03:42<00:00,  2.75it/s]


=== Perplexity on Skylion007/openwebtext, 4,996,727 tokens ===
main:        NLL/tok = 3.0915   PPL = 22.011
   combined[0]: NLL/tok = 3.0915   PPL = 22.011
 recurrence[1]: NLL/tok = 16.4901   PPL = 14506634.059
 recurrence[2]: NLL/tok = 19.1922   PPL = 216301394.080
 recurrence[3]: NLL/tok = 13.2760   PPL = 583017.906


## Combined summary across both datasets

In [13]:
combined_summary = {
    'checkpoint': CHECKPOINT_PATH,
    'context_len': int(MAX_CONTEXT_TOKENS),
    'seed': int(STREAM_SEED),
    f'{DATASET_NAME}/{DATASET_CONFIG}': {
        'n_scored_tokens': int(total_scored_tokens),
        'nll_per_tok': float(nll_main_sum / total_scored_tokens),
        'ppl': float(ppl_main),
        **({'ppl_per_recurrence': [float(math.exp(s / total_scored_tokens)) for s in nll_rec_sum]} if has_per_recurrence else {}),
    },
    f'{OWT_DATASET_NAME}': {
        'n_scored_tokens': int(owt_total_scored_tokens),
        'nll_per_tok': float(owt_nll_main_sum / owt_total_scored_tokens),
        'ppl': float(owt_ppl_main),
        **({'ppl_per_recurrence': [float(math.exp(s / owt_total_scored_tokens)) for s in owt_nll_rec_sum]} if has_per_recurrence else {}),
    },
}
combined_summary

{'checkpoint': '/raid/s3/opengptx/behzad_shomali/checkpoints/2026-05-06__02-32-43_d7e0557d0359234b/eid_2026-05-06__02-32-43_d7e0557d0359234b-seen_steps_27864-seen_tokens_6847856640-target_steps_27868-target_tokens_6848839680',
 'context_len': 2048,
 'seed': 42,
 'HuggingFaceFW/fineweb-edu/sample-10BT': {'n_scored_tokens': 4996727,
  'nll_per_tok': 2.984828354578407,
  'ppl': 19.7831062653119,
  'ppl_per_recurrence': [19.7831062653119,
   16571084.983678173,
   203469456.63322565,
   330184.8527376172]},
 'Skylion007/openwebtext': {'n_scored_tokens': 4996727,
  'nll_per_tok': 3.091542146707008,
  'ppl': 22.01099600075664,
  'ppl_per_recurrence': [22.01099600075664,
   14506634.059318358,
   216301394.0798331,
   583017.9063407846]}}